# AtrionNet: Synthetic Data Training Pipeline (Regime B)

This Colab notebook fully automates the isolated execution of the synthetic data pipeline. It is designed to prove that providing AtrionNet with enough overlapping data completely removes its baseline failure limits.

### Thesis Constraints Strictly Honored:
1. **Complete Isolation:** Training only touches the synthetic `data/` folder. It never imports `ludb_loader`.
2. **Controlled Evaluation:** The final trained model is tested on the exact same 15% real LUDB held-out test split (Seed 42) to provide a 1-to-1 comparison against Regime A (your ~65% F1 score model).

In [ ]:
# 1. Mount Google Drive and Enable GPU
from google.colab import drive
import torch

drive.mount('/content/drive')

if torch.cuda.is_available():
    print(f"\n✅ GPU Accelerated: {torch.cuda.get_device_name(0)}")
else:
    print("\n❌ ERROR: GPU not detected. Go to Runtime -> Change Runtime Type and select T4 GPU.")

In [ ]:
# 2. Navigate to the Synthetic Pipeline Directory
import os
import sys

# CHANGE THIS PATH if your project is named differently or in a subfolder!
PROJECT_PATH = '/content/drive/MyDrive/AtrionNet_Implementation/ml_component/synthetic_pipeline'

if not os.path.exists(PROJECT_PATH):
    print(f"❌ ERROR: Could not find folder at {PROJECT_PATH}")
    print("Please check your Google Drive to see exactly where you uploaded the 'ml_component' folder.")
else:
    os.chdir(PROJECT_PATH)
    print(f"📂 Successfully entered directory: {os.getcwd()}")
    
    # Ensure dependencies from the main project are installed
    !pip install -r ../requirements.txt

## Step 1: Generate the Isolated 1,500-Record Dataset
This script uses mathematical sine overlays and randomized pacing logic to generate physiological 3rd-Degree AV block anomalies, saving them locally to `synthetic_pipeline/data/`. This takes roughly 15-30 seconds.

In [ ]:
!python 01_synthetic_generator.py

## Step 2: Isolated Training (Regime B)
Trains a new AtrionNet instance from scratch for 150 epochs using exclusively the dataset generated in Step 1. The best model will be saved as `atrion_synthetic_best.pth`. This can take around 20-30 minutes on a T4 GPU.

In [ ]:
!python 02_train_synthetic_standalone.py

## Step 3: Formal Benchmark Against Real Patients
To prevent you from having to upload >1GB of dataset files, the script below will automatically download the LUDB dataset from PhysioNet directly into Colab's fast cloud memory.
Then, we evaluate the exclusively synthetic-trained weights against the authentic LUDB 15% test set.

In [ ]:
!python ../download_data.py
!python 03_evaluate_synthetic_vs_ludb.py